# Exercice 1 — Prédire le rachat à 12 mois : du dictionnaire à la note de synthèse

**Le contexte.** Au 31/12, la direction ALM veut la probabilité de **rachat (total ou partiel) dans les 12 mois** de chaque contrat en cours, pour fiabiliser la projection de liquidité et cibler la rétention. Vous disposez d'un portefeuille synthétique de ~20 000 observations *contrat × année* (2019 → 2023).

**Ce que vous allez faire** (chaque étape reprend un point de la session) :

| # | Étape | Notion de la session |
|---|---|---|
| Q1 | Débusquer la variable piégée | fuite de données, test de la date de disponibilité |
| Q2–Q3 | Split temporel, contrôle qualité | split passé → futur, classes déséquilibrées, accuracy trompeuse |
| Q4–Q5 | Baselines : taux segmentés, régression logistique | baseline actuarielle, pipeline sklearn, odds ratio |
| Q6 | Gradient boosting + réglage sur validation | boosting, hyperparamètres, jamais de réglage sur le test |
| Q7–Q8 | Tableau comparatif, seuil budgété | AUC, Brier, lift, précision/rappel |
| Q9 | Calibration | recalage de Platt, Brier avant/après |
| Q10 | Importance des variables | permutation, test du sens métier |
| Q11 | Note de 5 lignes + mini-quiz | livrable de la session |

> **Mode d'emploi.** Remplacez chaque `...` (ou `# TODO`) par votre code, puis exécutez la cellule suivante : une cellule `verifier(...)` vous dit tout de suite si c'est bon. Chaque question a un 💡 *indice* repliable. Les cellules « fournies » ne sont pas à modifier.

## Préparation (cellule fournie — à exécuter)

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings("ignore")
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss, precision_recall_curve
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

import hashlib
RES = {}
def verifier(qid, test, ok="Bravo !", ko="Pas encore — relis l'énoncé ou ouvre l'indice."):
    """Auto-correction : exécute `test()` et enregistre le résultat de la question `qid`."""
    try:
        res = bool(test()); detail = ""
    except Exception as e:
        res = False; detail = f" [{type(e).__name__} : {e}]"
    RES[qid] = res
    print(("✅ " + ok) if res else ("❌ " + ko + detail))

def qcm(qid, reponse, empreinte):
    ok = hashlib.sha256(f"{qid}:{str(reponse).strip().upper()}".encode()).hexdigest()[:10] == empreinte
    RES[qid] = ok
    print("✅ Bonne réponse !" if ok else "❌ Ce n'est pas la bonne réponse — relis la slide correspondante.")

def bilan(attendus):
    print("BILAN DE L'EXERCICE")
    for q in attendus: print(f"  {q:<6}", "✅" if RES.get(q) else "⬜ à faire / à revoir")
    print(f"\nScore : {sum(RES.get(q, False) for q in attendus)}/{len(attendus)}")
import numpy as np, pandas as pd

def generer_portefeuille(n=20000, seed=42):
    """Portefeuille vie synthétique : 1 ligne = 1 contrat observé au 31/12 de `annee_obs`."""
    rng = np.random.default_rng(seed)
    annee_obs = rng.choice([2019, 2020, 2021, 2022, 2023], size=n)
    age = np.clip(rng.normal(52, 14, n), 22, 88).round()
    anciennete = np.clip(rng.gamma(2.2, 3.0, n), 0, 25).round(1)
    garantie = rng.choice(["Euro", "UC", "Eurocroissance"], size=n, p=[.55, .35, .10])
    canal = rng.choice(["Agent", "Courtier", "Direct"], size=n, p=[.5, .3, .2])
    encours = np.exp(rng.normal(10.3, 1.0, n)).round(-2)               # € (log-normal)
    versements_12m = np.where(rng.random(n) < .45, 0, encours * rng.uniform(.0, .12, n)).round(-1)
    nb_arbitrages_12m = rng.poisson(0.6, n)
    nb_rachats_passes = rng.poisson(0.25, n)
    ecart_taux = np.round(rng.normal(0, 0.8, n) - 0.15 * (annee_obs == 2022) * 2, 2)  # taux marché - taux servi (pts)
    z = (-4.3 + 0.9*ecart_taux + 0.6*np.maximum(ecart_taux, 0)**2 + 1.6*((anciennete >= 8) & (anciennete < 10)) + 1.3*((age > 62) & (encours > 80000)) - 0.9*((age < 35) & (garantie == 'Euro'))
         + 0.015*(age - 52) - 0.25*np.log(encours/30000)
         + 0.25*nb_arbitrages_12m + 0.35*nb_rachats_passes
         + 0.25*(garantie == "UC") + 0.3*(canal == "Direct") + 0.35*(annee_obs == 2022))
    p = 1/(1+np.exp(-z))
    rachat = (rng.random(n) < p).astype(int)
    motif = np.where(rachat == 1, rng.choice(["Besoin de liquidité", "Arbitrage vers autre support",
                                              "Décès/Dépendance", "Fiscalité"], size=n), None)
    df = pd.DataFrame(dict(contrat_id=np.arange(1, n+1), annee_obs=annee_obs, age=age, anciennete=anciennete,
        garantie=garantie, canal=canal, encours=encours, versements_12m=versements_12m,
        nb_arbitrages_12m=nb_arbitrages_12m, nb_rachats_passes=nb_rachats_passes, ecart_taux=ecart_taux,
        motif_rachat=motif, rachat_12m=rachat))
    df.loc[rng.random(n) < .06, "encours"] = np.nan          # valeurs manquantes volontaires
    df.loc[rng.random(n) < .05, "canal"] = np.nan
    return df

df = generer_portefeuille()
print(f"{len(df):,} observations contrat × année · taux de rachat global : {df['rachat_12m'].mean():.1%}")
df.head()

20,000 observations contrat × année · taux de rachat global : 6.2%


   contrat_id  annee_obs    age  anciennete garantie     canal    encours  \
0           1       2019 26.000       3.400       UC    Direct 18,000.000   
1           2       2022 68.000       1.700       UC     Agent 32,100.000   
2           3       2022 69.000       4.600       UC     Agent 29,400.000   
3           4       2021 45.000       9.900     Euro  Courtier 17,200.000   
4           5       2021 59.000       1.200       UC     Agent 20,700.000   

   versements_12m  nb_arbitrages_12m  nb_rachats_passes  ecart_taux  \
0       1,620.000                  1                  3      -0.990   
1       2,520.000                  1                  0       0.320   
2       2,900.000                  0                  1      -0.620   
3       1,470.000                  0                  0      -0.830   
4           0.000                  0                  0       0.380   

                   motif_rachat  rachat_12m  
0                           NaN           0  
1  Arbitrage vers 

### Dictionnaire de données
| Variable | Description | Type |
|---|---|---|
| `contrat_id` | Identifiant du contrat | identifiant |
| `annee_obs` | Année d'observation : on prédit au **31/12** de cette année | période |
| `age` | Âge de l'assuré au 31/12 | numérique |
| `anciennete` | Ancienneté du contrat (années) | numérique |
| `garantie` | Euro / UC / Eurocroissance | catégorielle |
| `canal` | Agent / Courtier / Direct *(≈5 % manquant)* | catégorielle |
| `encours` | Provision mathématique au 31/12 (€) *(≈6 % manquant)* | numérique |
| `versements_12m` | Versements sur les 12 mois **précédant** le 31/12 (€) | numérique |
| `nb_arbitrages_12m` | Arbitrages sur les 12 mois **précédant** le 31/12 | numérique |
| `nb_rachats_passes` | Rachats antérieurs au 31/12 | numérique |
| `ecart_taux` | Taux de marché − taux servi au 31/12 (en points) | numérique |
| `motif_rachat` | Motif déclaré par le client lors de son rachat | catégorielle |
| **`rachat_12m`** | **CIBLE** : 1 si rachat dans les 12 mois suivant le 31/12 | binaire |

## Q1 · Le test de la date de disponibilité 
**Notions :** fuite de données · *Cette information était-elle connue au 31/12 ?*

1. Parcourez le dictionnaire : une variable n'existerait **pas** à la date de prédiction. Laquelle ?
2. Mesurez son pouvoir « prédictif » : calculez l'AUC obtenue en utilisant **uniquement** l'indicatrice `df["motif_rachat"].notna()` comme score pour prédire `rachat_12m`. Une AUC proche de 0,99 = *alarme, pas victoire*.
3. Construisez la liste `EXCLUDED` des colonnes à **ne pas** utiliser comme variables explicatives (identifiant, cible, et la variable piégée).


<details><summary>💡 <b>Indice</b></summary>

- `roc_auc_score(y_vrai, score)` ; ici le score est `df["motif_rachat"].notna()` (True/False).
- Une colonne renseignée *si et seulement si* l'événement a eu lieu contient la cible elle-même.

</details>

In [2]:
auc_motif = roc_auc_score(df["rachat_12m"], df["motif_rachat"].notna())
print(f"AUC de la seule présence de 'motif_rachat' : {auc_motif:.3f}")

EXCLUDED = ["contrat_id", "motif_rachat", "rachat_12m"]

AUC de la seule présence de 'motif_rachat' : 1.000


In [3]:
LEGIT = ["age", "anciennete", "encours", "versements_12m", "nb_arbitrages_12m", "nb_rachats_passes", "ecart_taux", "garantie", "canal"]
verifier("Q1a", lambda: auc_motif > 0.99, "AUC ≈ 1 : la variable contient la réponse → fuite caractérisée.")
verifier("Q1b", lambda: {"motif_rachat", "rachat_12m", "contrat_id"} <= set(EXCLUDED) and not (set(EXCLUDED) & set(LEGIT)),
         "EXCLUDED est correct : fuite + cible + identifiant exclus, aucune variable légitime sacrifiée.",
         "Il faut exclure l'identifiant, la cible et la variable piégée — mais aucune variable légitime.")
FEATURES = [c for c in df.columns if c not in EXCLUDED and c != "annee_obs"]
TARGET = "rachat_12m"
print("Variables retenues :", FEATURES)

✅ AUC ≈ 1 : la variable contient la réponse → fuite caractérisée.
✅ EXCLUDED est correct : fuite + cible + identifiant exclus, aucune variable légitime sacrifiée.
Variables retenues : ['age', 'anciennete', 'garantie', 'canal', 'encours', 'versements_12m', 'nb_arbitrages_12m', 'nb_rachats_passes', 'ecart_taux']


**Correction commentée :** `motif_rachat` n'est renseigné **qu'après** le rachat : en production, au 31/12, elle serait toujours vide. Elle explose le score en test et s'effondre en production. On exclut aussi l'identifiant (sans signal, source de sur-apprentissage) et, bien sûr, la cible.

## Q2 · Le split temporel 
**Notion :** *on entraîne sur le passé, on évalue sur le futur — comme en production.*

Découpez `df` selon `annee_obs` :
- **train** : 2019, 2020, 2021 · **valid** : 2022 (réglages) · **test** : 2023 (évaluation finale, utilisée **une seule fois**).


<details><summary>💡 <b>Indice</b></summary>

Un filtre pandas : `df[df['annee_obs'] <= 2021]`.

</details>

In [4]:
train = df[df["annee_obs"] <= 2021]
valid = df[df["annee_obs"] == 2022]
test  = df[df["annee_obs"] == 2023]

In [5]:
verifier("Q2", lambda: len(train) + len(valid) + len(test) == len(df)
         and train["annee_obs"].max() < valid["annee_obs"].min() < test["annee_obs"].min(),
         "Split temporel valide : aucune observation perdue, le futur n'est jamais dans l'apprentissage.")
X_train, y_train = train[FEATURES], train[TARGET]
X_valid, y_valid = valid[FEATURES], valid[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]
print(f"train {len(train):,} | valid {len(valid):,} | test {len(test):,}")

✅ Split temporel valide : aucune observation perdue, le futur n'est jamais dans l'apprentissage.
train 12,123 | valid 4,009 | test 3,868


## Q3 · Classes déséquilibrées : l'accuracy trompeuse
**Notion :** *avec 6 % de rachats, prédire « personne ne rachète » donne 94 % d'accuracy — et zéro valeur.*

1. Calculez `taux_rachat_train` (taux de rachat dans le train).
2. Calculez `acc_triviale` : l'accuracy sur le **test** du modèle qui prédit toujours « pas de rachat ».


<details><summary>💡 <b>Indice</b></summary>

L'accuracy du modèle trivial vaut `1 - y_test.mean()`.

</details>

In [6]:
taux_rachat_train = y_train.mean()
acc_triviale = 1 - y_test.mean()
print(f"Taux de rachat (train) : {taux_rachat_train:.1%} | accuracy du modèle « personne ne rachète » : {acc_triviale:.1%}")

Taux de rachat (train) : 6.2% | accuracy du modèle « personne ne rachète » : 93.3%


In [7]:
verifier("Q3", lambda: abs(taux_rachat_train - y_train.mean()) < 1e-9 and abs(acc_triviale - (1 - y_test.mean())) < 1e-9,
         "Un modèle sans aucune valeur affiche ~94 % d'accuracy : on évaluera donc avec AUC, Brier, lift et précision/rappel.")

✅ Un modèle sans aucune valeur affiche ~94 % d'accuracy : on évaluera donc avec AUC, Brier, lift et précision/rappel.


## Q4 · Votre première métrique métier : le lift
**Notion :** *« les 10 % de contrats les plus à risque concentrent 35 % des rachats »* — la métrique qui parle à l'ALM.

Complétez `lift_top(y, p, frac)` = (taux de rachat observé parmi les `frac` des contrats **les mieux classés**) ÷ (taux de rachat global).


<details><summary>💡 <b>Indice</b></summary>

1. `k = int(np.ceil(frac * len(y)))` contrats à retenir.
2. `np.argsort(-p)[:k]` donne les indices des k plus fortes probabilités.
3. Lift = `y[idx].mean() / y.mean()`.

</details>

In [8]:
def lift_top(y, p, frac=0.10):
    y, p = np.asarray(y), np.asarray(p)
    k = int(np.ceil(frac * len(y)))
    idx = np.argsort(-p)[:k]
    return y[idx].mean() / y.mean()

In [9]:
_y = np.zeros(100); _y[[0,1,2,3,4,50,51,52,53,54]] = 1      # 10 % de rachats, dont 5 dans le top 10
verifier("Q4", lambda: abs(lift_top(_y, np.linspace(1, 0, 100), 0.10) - 5.0) < 1e-9,
         "lift = 5× : le top 10 % contient 50 % de rachats contre 10 % en moyenne.")

def evaluer(y, p):
    """Fournie : les trois métriques du tableau comparatif."""
    return {"AUC": roc_auc_score(y, p), "Brier": brier_score_loss(y, p), "Lift top 10 %": lift_top(y, p, 0.10)}

✅ lift = 5× : le top 10 % contient 50 % de rachats contre 10 % en moyenne.


## Q5 · La baseline actuarielle : taux segmentés
**Notion :** *tout modèle complexe devra démontrer un gain net face à la baseline.*

La fonction `add_segment` (fournie) crée un segment **tranche d'âge × garantie** (4 × 3 = 12 cellules). Apprenez sur `train_s` :
1. `taux_seg` : le taux de rachat moyen par segment.
2. `taux_global` : le taux moyen du train (pour les segments jamais vus).
3. `p_seg_valid` et `p_seg_test` : la probabilité prédite = taux du segment (`.map`), `taux_global` si segment inconnu (`.fillna`).


<details><summary>💡 <b>Indice</b></summary>

`train_s.groupby('segment')['rachat_12m'].mean()` puis `valid_s['segment'].map(taux_seg).fillna(taux_global)`.

</details>

In [10]:
def add_segment(d):
    d = d.copy()
    d["tranche_age"] = pd.cut(d["age"], [0, 40, 55, 70, 120], labels=["<40", "40-55", "55-70", "70+"])
    d["segment"] = d["tranche_age"].astype(str) + " | " + d["garantie"]
    return d
train_s, valid_s, test_s = map(add_segment, (train, valid, test))

taux_seg = train_s.groupby("segment")["rachat_12m"].mean()
taux_global = train_s["rachat_12m"].mean()
p_seg_valid = valid_s["segment"].map(taux_seg).fillna(taux_global)
p_seg_test = test_s["segment"].map(taux_seg).fillna(taux_global)

In [11]:
verifier("Q5", lambda: len(taux_seg) == 12 and len(p_seg_test) == len(test) and p_seg_test.notna().all()
         and 0.55 < roc_auc_score(y_valid, p_seg_valid) < 0.85,
         f"Baseline segmentée prête (AUC validation = {roc_auc_score(y_valid, p_seg_valid):.3f}).")
# Rappel de prudence statistique : les taux ne se lisent jamais sans l'effectif du segment
pd.DataFrame({"taux": taux_seg, "effectif": train_s.groupby("segment").size()}).sort_values("taux", ascending=False)

✅ Baseline segmentée prête (AUC validation = 0.613).


                        taux  effectif
segment                               
70+ | UC               0.102       400
70+ | Euro             0.082       648
55-70 | UC             0.082      1274
70+ | Eurocroissance   0.073       124
55-70 | Euro           0.067      2076
40-55 | UC             0.067      1650
40-55 | Euro           0.058      2598
<40 | UC               0.057       857
40-55 | Eurocroissance 0.055       492
<40 | Eurocroissance   0.052       232
55-70 | Eurocroissance 0.034       383
<40 | Euro             0.031      1389

## Q6 · Régression logistique dans un pipeline
**Notions :** régression logistique · pipeline sklearn (μ et σ appris sur le train uniquement) · odds ratio.

Le préprocesseur `prep` est fourni (imputation + standardisation + one-hot). À vous de :
1. Construire `pipe_logit = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=1000))])`, l'entraîner sur le **train**.
2. Calculer `p_logit_valid`, `p_logit_test` (probabilité de la classe 1 : `predict_proba(...)[:, 1]`).
3. Calculer `odds_ratio = np.exp(coefs)` : de combien est multipliée la *cote* de rachat quand la variable (standardisée) augmente d'un écart-type ?


<details><summary>💡 <b>Indice</b></summary>

Un `Pipeline` s'utilise comme un modèle : `.fit(X_train, y_train)` puis `.predict_proba(X)`.

</details>

In [12]:
NUM = ["age", "anciennete", "encours", "versements_12m", "nb_arbitrages_12m", "nb_rachats_passes", "ecart_taux"]
CAT = ["garantie", "canal"]
prep = ColumnTransformer([
    ("n", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM),
    ("c", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Inconnu")),
                    ("oh", OneHotEncoder(handle_unknown="ignore"))]), CAT)])

pipe_logit = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=1000))])
pipe_logit.fit(X_train, y_train)
p_logit_valid = pipe_logit.predict_proba(X_valid)[:, 1]
p_logit_test = pipe_logit.predict_proba(X_test)[:, 1]

In [13]:
verifier("Q6a", lambda: roc_auc_score(y_valid, p_logit_valid) > roc_auc_score(y_valid, p_seg_valid),
         f"La logistique (AUC valid {roc_auc_score(y_valid, p_logit_valid):.3f}) bat les taux segmentés.")
coefs = pd.Series(pipe_logit["clf"].coef_[0], index=pipe_logit["prep"].get_feature_names_out())
odds_ratio = np.exp(coefs)

✅ La logistique (AUC valid 0.793) bat les taux segmentés.


In [14]:
verifier("Q6b", lambda: abs(odds_ratio["n__ecart_taux"] - np.exp(coefs["n__ecart_taux"])) < 1e-9 and odds_ratio["n__ecart_taux"] > 1,
         "OR(écart de taux) > 1 : plus le taux servi est inférieur au marché, plus la cote de rachat augmente — cohérent avec la théorie du rachat.")
odds_ratio.sort_values(ascending=False).head(6)

✅ OR(écart de taux) > 1 : plus le taux servi est inférieur au marché, plus la cote de rachat augmente — cohérent avec la théorie du rachat.


n__ecart_taux          3.805
c__canal_Inconnu       1.306
n__age                 1.305
c__garantie_UC         1.275
n__nb_arbitrages_12m   1.182
n__anciennete          1.139
dtype: float64

## Q7 · Gradient boosting et réglage sur la validation
**Notions :** boosting séquentiel · hyperparamètres (profondeur, pas ν) · *le test ne sert qu'une fois, à la fin*.

1. Pour chaque profondeur `depth` de `[2, 3, 4]`, entraînez `GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=depth, random_state=0)` dans un pipeline avec `prep`, et notez l'**AUC sur la validation**.
2. `best_depth` = profondeur au meilleur AUC **validation** (⚠️ pas sur le test).
3. Ré-entraînez avec `best_depth` : `pipe_gb`, puis `p_gb_valid`, `p_gb_test`.


<details><summary>💡 <b>Indice</b></summary>

`best_depth = max(auc_valid_par_depth, key=auc_valid_par_depth.get)` renvoie la clé du plus grand AUC.

</details>

In [15]:
auc_valid_par_depth = {}
for depth in [2, 3, 4]:
    pipe = Pipeline([("prep", prep),
                     ("clf", GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=depth, random_state=0))])
    pipe.fit(X_train, y_train)
    auc_valid_par_depth[depth] = roc_auc_score(y_valid, pipe.predict_proba(X_valid)[:, 1])
print(auc_valid_par_depth)

best_depth = max(auc_valid_par_depth, key=auc_valid_par_depth.get)
pipe_gb = Pipeline([("prep", prep),
                    ("clf", GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=best_depth, random_state=0))])
pipe_gb.fit(X_train, y_train)
p_gb_valid = pipe_gb.predict_proba(X_valid)[:, 1]
p_gb_test = pipe_gb.predict_proba(X_test)[:, 1]

{2: 0.814984519317097, 3: 0.8145018583926149, 4: 0.8072163434319028}


In [16]:
verifier("Q7", lambda: best_depth == max(auc_valid_par_depth, key=auc_valid_par_depth.get)
         and roc_auc_score(y_valid, p_gb_valid) > roc_auc_score(y_valid, p_logit_valid),
         f"Profondeur retenue : {best_depth}, choisie sur la validation. Le boosting dépasse la logistique en validation.")

✅ Profondeur retenue : 2, choisie sur la validation. Le boosting dépasse la logistique en validation.


## Q8 · Tableau comparatif et seuil « budget d'actions »
**Notions :** comparer honnêtement (test temporel uniquement) · précision / rappel · *le seuil est un budget*.

**8a.** Construisez `tableau` : une ligne par modèle (*Taux segmentés*, *Régression logistique*, *Gradient boosting*), les métriques de `evaluer(y_test, p_...)` en colonnes.

**8b.** L'équipe de rétention peut contacter **300 contrats** (`BUDGET = 300`). Complétez `precision_at_k` (part de vrais rachats parmi les k contrats les mieux classés) et `recall_at_k` (part des rachats totaux captés), puis appliquez-les au gradient boosting.


<details><summary>💡 <b>Indice</b></summary>

- 8a : `pd.DataFrame({"Taux segmentés": evaluer(...), ...}).T`
- 8b : même logique que `lift_top` : `idx = np.argsort(-p)[:k]`, puis `y[idx].mean()` (précision) et `y[idx].sum() / y.sum()` (rappel).

</details>

In [17]:
tableau = pd.DataFrame({
    "Taux segmentés": evaluer(y_test, p_seg_test),
    "Régression logistique": evaluer(y_test, p_logit_test),
    "Gradient boosting": evaluer(y_test, p_gb_test)}).T
BUDGET = 300

def precision_at_k(y, p, k):
    y, p = np.asarray(y), np.asarray(p)
    idx = np.argsort(-p)[:k]
    return y[idx].mean()

def recall_at_k(y, p, k):
    y, p = np.asarray(y), np.asarray(p)
    idx = np.argsort(-p)[:k]
    return y[idx].sum() / y.sum()

prec_gb = precision_at_k(y_test, p_gb_test, BUDGET)
rec_gb = recall_at_k(y_test, p_gb_test, BUDGET)
print(f"Gradient boosting, {BUDGET} contrats contactés : précision {prec_gb:.1%} | rappel {rec_gb:.1%} | lift {prec_gb / y_test.mean():.1f}×")
tableau

Gradient boosting, 300 contrats contactés : précision 37.0% | rappel 42.5% | lift 5.5×


                        AUC  Brier  Lift top 10 %
Taux segmentés        0.539  0.063          1.034
Régression logistique 0.791  0.053          4.327
Gradient boosting     0.827  0.051          4.710

In [18]:
verifier("Q8a", lambda: tableau.shape == (3, 3) and set(tableau.columns) == {"AUC", "Brier", "Lift top 10 %"},
         "Tableau comparatif correct (3 modèles × 3 métriques).")
_yt = np.array([1, 0, 1, 0, 0, 0, 1, 0, 0, 0]); _pt = np.array([.9, .8, .7, .6, .5, .4, .3, .2, .1, 0])
verifier("Q8b", lambda: abs(precision_at_k(_yt, _pt, 3) - 2/3) < 1e-9 and abs(recall_at_k(_yt, _pt, 3) - 2/3) < 1e-9,
         "précision@k et rappel@k corrects.")

# Courbes précision–rappel (fournies)
for nom, p in [("Taux segmentés", p_seg_test), ("Logistique", p_logit_test), ("Gradient boosting", p_gb_test)]:
    pr, rc, _ = precision_recall_curve(y_test, p); plt.plot(rc, pr, label=nom)
plt.axhline(y_test.mean(), ls=":", c="grey", label="niveau de base"); plt.xlabel("Rappel"); plt.ylabel("Précision")
plt.title("Courbes précision–rappel (test 2023)"); plt.legend(); plt.show()

✅ Tableau comparatif correct (3 modèles × 3 métriques).
✅ précision@k et rappel@k corrects.


<Figure size 640x480 with 1 Axes>

**Correction commentée :** l'ordre attendu est taux segmentés < logistique < boosting sur l'AUC, le Brier (↓) et le lift. Le point à argumenter est le *coût/bénéfice* : quelques points d'AUC sur un modèle moins lisible et plus long à valider. Avec 300 contrats sur ~3 900, on contacte ~8 % du portefeuille : le rappel indique la part des rachats de l'année captée, à comparer au ~8 % d'un tirage aléatoire (c'est exactement le lift).

## Q9 · Calibration : rendre la probabilité utilisable
**Notions :** calibration · Brier avant/après recalage · *la projection de liquidité consomme des probabilités, pas des rangs.*

1. **Nombre de rachats attendu vs observé en test** : `attendus_gb = p_gb_test.sum()` et `observes = y_test.sum()`.
2. **Recalage de Platt appris sur la validation** : régression logistique à une variable, `logit(p)`, entraînée sur `(logit(p_gb_valid), y_valid)`. Appliquez-la au test → `p_gb_cal`.
3. Comparez `brier_avant` et `brier_apres` (sur le test).


<details><summary>💡 <b>Indice</b></summary>

- `logit(p) = np.log(p / (1 - p))` : bornez d'abord p avec `np.clip(p, 1e-6, 1 - 1e-6)`.
- `platt.fit(logit(p_valid).reshape(-1, 1), y_valid)` puis `platt.predict_proba(logit(p_test).reshape(-1, 1))[:, 1]`.

</details>

In [19]:
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

attendus_gb = p_gb_test.sum()
observes = y_test.sum()

platt = LogisticRegression()
platt.fit(logit(p_gb_valid).reshape(-1, 1), y_valid)
p_gb_cal = platt.predict_proba(logit(p_gb_test).reshape(-1, 1))[:, 1]

brier_avant = brier_score_loss(y_test, p_gb_test)
brier_apres = brier_score_loss(y_test, p_gb_cal)
print(f"Rachats attendus {attendus_gb:.0f} vs observés {observes} | Brier avant {brier_avant:.5f} → après {brier_apres:.5f}")

Rachats attendus 249 vs observés 261 | Brier avant 0.05097 → après 0.05125


In [20]:
verifier("Q9a", lambda: abs(attendus_gb - p_gb_test.sum()) < 1e-6 and observes == y_test.sum(), "Attendu vs observé calculés.")
verifier("Q9b", lambda: len(p_gb_cal) == len(test) and 0 <= p_gb_cal.min() and p_gb_cal.max() <= 1
         and abs(brier_apres - brier_score_loss(y_test, p_gb_cal)) < 1e-12 and brier_apres <= brier_avant + 0.001,
         "Recalage réalisé : le Brier ne se dégrade pas et le niveau de probabilité est cohérent avec le terrain.")

# Courbes de calibration (fournies)
for nom, p in [("avant recalage", p_gb_test), ("après recalage", p_gb_cal)]:
    fo, fp = calibration_curve(y_test, p, n_bins=8, strategy="quantile"); plt.plot(fp, fo, "o-", label=nom)
plt.plot([0, .3], [0, .3], "k:", label="calibration parfaite"); plt.xlabel("Probabilité prédite"); plt.ylabel("Fréquence observée")
plt.title("Calibration du gradient boosting (test 2023)"); plt.legend(); plt.show()

✅ Attendu vs observé calculés.
✅ Recalage réalisé : le Brier ne se dégrade pas et le niveau de probabilité est cohérent avec le terrain.


<Figure size 640x480 with 1 Axes>

**Correction commentée :** un bon classement (AUC) ne garantit pas le bon *niveau* de probabilité. Comparer `Σp` au nombre de rachats observés est le test actuariel le plus parlant : c'est lui qui conditionne le flux de liquidité projeté. Ici le boosting est déjà bien calibré en niveau (~250 rachats attendus pour ~260 observés) : le recalage appris sur la validation (jamais sur le test) ne change presque rien au Brier — **et c'est une conclusion valide** : on documente qu'il n'est pas nécessaire sur ce périmètre. À vérifier aussi **par génération** et par tranche d'encours.

## Q10 · Interpréter : importance par permutation et sens métier
**Notion :** *on mélange une variable et on mesure la perte de performance* — robuste et agnostique du modèle.

Calculez l'importance par permutation de `pipe_gb` sur la **validation** (`scoring="roc_auc"`, `n_repeats=5`, `random_state=0`) puis rangez-la dans `imp` (Series indexée par `FEATURES`, triée décroissante). Y a-t-il une variable **absurde** en tête ?


<details><summary>💡 <b>Indice</b></summary>

`perm = permutation_importance(pipe_gb, X_valid, y_valid, scoring="roc_auc", n_repeats=5, random_state=0)` ; `perm.importances_mean` contient les valeurs.

</details>

In [21]:
perm = permutation_importance(pipe_gb, X_valid, y_valid, scoring="roc_auc", n_repeats=5, random_state=0)
imp = pd.Series(perm.importances_mean, index=FEATURES).sort_values(ascending=False)
imp

ecart_taux          0.182
anciennete          0.050
age                 0.034
encours             0.017
versements_12m      0.008
garantie            0.003
nb_rachats_passes   0.003
canal               0.002
nb_arbitrages_12m   0.002
dtype: float64

In [22]:
verifier("Q10", lambda: len(imp) == len(FEATURES) and "ecart_taux" in list(imp.index[:3]),
         "L'écart taux servi / marché figure en tête : cohérent avec la théorie du rachat (arbitrage financier). Aucune variable absurde → pas de fuite résiduelle.")
imp.sort_values().plot.barh(title="Importance par permutation (perte d'AUC, validation)"); plt.show()

✅ L'écart taux servi / marché figure en tête : cohérent avec la théorie du rachat (arbitrage financier). Aucune variable absurde → pas de fuite résiduelle.


<Figure size 640x480 with 1 Axes>

## Q11 · Livrable : la note de 5 lignes + mini-quiz
**Rédigez** (dans la cellule Markdown ci-dessous) la note destinée au comité modèles : **(1)** modèle recommandé · **(2)** justification chiffrée (AUC, Brier, lift, calibration vs baseline) · **(3)** limite principale · **(4)** contrôle à ajouter · **(5)** variable écartée et pourquoi.

> Rappel de la session : *le meilleur score en test n'est pas automatiquement le meilleur modèle en production* — stabilité, calibration et lisibilité comptent.

*Votre note :*

**Exemple de note attendue :** (1) Gradient boosting (profondeur retenue sur validation), après recalage de Platt. (2) Sur le test 2023, il surpasse la logistique et les taux segmentés en AUC, Brier et lift ; avec 300 contacts il capte une part des rachats plusieurs fois supérieure au hasard. (3) Moins lisible et plus long à valider que la logistique ; calibration non vérifiée par génération ni par tranche d'encours. (4) Suivi mensuel de la calibration (Σp vs rachats observés) et revue de la stabilité des importances. (5) `motif_rachat` écartée : variable postérieure à l'événement (fuite).

**Mini-quiz** — répondez avec `A`, `B` ou `C` dans les variables `rep_...`.

**Q11a.** Pourquoi un split temporel plutôt qu'aléatoire ?
- **A** — Il est plus rapide à calculer.
- **B** — Il reproduit l'usage réel (apprendre du passé, prédire le futur) et évite un score gonflé par le mélange des périodes.
- **C** — Il rééquilibre les classes.

**Q11b.** Après l'ajout d'une nouvelle variable, l'AUC test passe de 0,82 à 0,99. Que faites-vous ?
- **A** — Je déploie : c'est une victoire.
- **B** — J'applique le test de la date de disponibilité : c'est probablement une fuite.
- **C** — Je réduis la profondeur des arbres.

**Q11c.** Pourquoi la calibration est-elle une « condition d'usage » en assurance vie ?
- **A** — Elle améliore mécaniquement l'AUC.
- **B** — Elle rend le modèle plus lisible.
- **C** — La projection de liquidité consomme des probabilités, pas des rangs.

In [23]:
rep_q11a, rep_q11b, rep_q11c = "B", "B", "C"
qcm("Q11a", rep_q11a, "fd7c3ab0ab")
qcm("Q11b", rep_q11b, "cb7694bd44")
qcm("Q11c", rep_q11c, "f76c6960ed")

✅ Bonne réponse !
✅ Bonne réponse !
✅ Bonne réponse !


##  Bilan

In [24]:
bilan(["Q1a", "Q1b", "Q2", "Q3", "Q4", "Q5", "Q6a", "Q6b", "Q7", "Q8a", "Q8b", "Q9a", "Q9b", "Q10", "Q11a", "Q11b", "Q11c"])

BILAN DE L'EXERCICE
  Q1a    ✅
  Q1b    ✅
  Q2     ✅
  Q3     ✅
  Q4     ✅
  Q5     ✅
  Q6a    ✅
  Q6b    ✅
  Q7     ✅
  Q8a    ✅
  Q8b    ✅
  Q9a    ✅
  Q9b    ✅
  Q10    ✅
  Q11a   ✅
  Q11b   ✅
  Q11c   ✅

Score : 17/17
